# Prototype build unique ds

Involved ds
- copernicus 1
- copernicus 2
- gfw ais
- gfw sar

Target ds columns
- date 
- latitude
- longitude
- chl
- thetao
- number of ais vessels
- number of ais gear 1
- number of ais gear 2
- ...
- number of sar vessels 

Filter: 
- optional filter to remove the Svn data


In [1]:
import pandas as pd
import os 
import numpy as np

# Set

In [2]:
year = 2024

input_path = "../data/interim"
    # cmems_mod_med_phy-tem_anfc_4.2km_P1D-m
    # cmems_mod_med_bgc-pft_anfc_4.2km_P1D-m
cpr_product_name_1 = 'cmems_mod_med_bgc-pft_anfc_4.2km_P1D-m'
cpr_product_name_2 = 'cmems_mod_med_phy-tem_anfc_4.2km_P1D-m'
gfw_ais_product_name = 'ais_high_daily_ts'
gfw_sar_product_name = 's2_vessel_detections'

REMOVE_SVN_VESSELS = False

keys = ['date', 'latitude', 'longitude']
spatial_keys = ['latitude', 'longitude']
cpr_marine_var_1 = 'chl'
cpr_marine_var_2 = 'thetao'

# Automatic 

cpr_df_name_1 = f"cleaned_{cpr_product_name_1}_{year}.parquet"
cpr_df_name_2 = f"cleaned_{cpr_product_name_2}_{year}.parquet"
gfw_ais_df_name = f"cleaned_{gfw_ais_product_name}_{year}.parquet"
gfw_sar_df_name = f"cleaned_{gfw_sar_product_name}_{year}.parquet"

cpr_1_input = os.path.join(input_path, cpr_df_name_1)
cpr_2_input = os.path.join(input_path, cpr_df_name_2)
gfw_ais_input = os.path.join(input_path, gfw_ais_df_name)
gfw_sar_input = os.path.join(input_path, gfw_sar_df_name)

# Import 

In [3]:
cpr_1 = pd.read_parquet(cpr_1_input)
cpr_2 = pd.read_parquet(cpr_2_input)
gfw_ais = pd.read_parquet(gfw_ais_input)
gfw_sar = pd.read_parquet(gfw_sar_input)

In [4]:
gfw_sar.head()

,date,latitude,longitude,ssvid,speed_kn_inferred,heading_deg_inferred,length_m_inferred,presence_score
0,2024-05-27,45.623222,13.633012,218025010,3.03,309.488462,17.5,0.547
1,2024-05-09,45.518940,13.562941,218020360,0.63,194.323945,22.7,0.675
2,2024-05-09,45.471623,13.488535,None,1.44,357.043959,24.4,0.808
3,2024-05-09,45.567544,13.643718,None,3.98,354.179170,21.0,0.925
4,2024-05-09,45.490410,13.584911,203245488,0.24,76.203387,21.4,0.747


# Svn filter (optional)


In [5]:
if REMOVE_SVN_VESSELS:
    # write filter 
    print()

# Build dataset

### Structure the df

In [6]:
df = cpr_1.drop_duplicates(subset=keys)[keys].copy()

In [7]:
print(df.shape[0])
df.head()

27084


,date,latitude,longitude
0,2024-01-01,45.479168,13.166667
1,2024-01-02,45.479168,13.166667
2,2024-01-03,45.479168,13.166667
3,2024-01-04,45.479168,13.166667
4,2024-01-05,45.479168,13.166667


### Enrich df with coperncius data 

In [8]:
df = pd.merge(
    left = df,
    right = cpr_1,
    how = 'left',
    on = keys
)

In [9]:
cols_to_save = keys + [cpr_marine_var_1] 
cols_to_drop = [col for col in df.columns if col not in cols_to_save]
df = df.drop(columns=cols_to_drop)

In [10]:
print(len(df))
df.head()

27084


,date,latitude,longitude,chl
0,2024-01-01,45.479168,13.166667,0.432612
1,2024-01-02,45.479168,13.166667,0.409951
2,2024-01-03,45.479168,13.166667,0.387921
3,2024-01-04,45.479168,13.166667,0.367288
4,2024-01-05,45.479168,13.166667,0.347252


In [11]:
df = pd.merge(
    left = df,
    right = cpr_2,
    how = 'left',
    on = keys
)

In [12]:
cols_to_save = keys + [cpr_marine_var_1, cpr_marine_var_2] 
cols_to_drop = [col for col in df.columns if col not in cols_to_save]
df = df.drop(columns=cols_to_drop)

In [13]:
print(len(df))
df.head()

27084


,date,latitude,longitude,chl,thetao
0,2024-01-01,45.479168,13.166667,0.432612,13.304394
1,2024-01-02,45.479168,13.166667,0.409951,13.248711
2,2024-01-03,45.479168,13.166667,0.387921,13.063446
3,2024-01-04,45.479168,13.166667,0.367288,12.772483
4,2024-01-05,45.479168,13.166667,0.347252,13.004619


### Enrich sar with target_lat, target_lon

In [14]:
sar_len_before = len(gfw_sar)

In [15]:
sar_unq_coords = gfw_sar.drop_duplicates(subset=spatial_keys)[spatial_keys].copy()

In [16]:
df_unq_coords = df.drop_duplicates(subset=spatial_keys)[spatial_keys].copy()

In [17]:
def project_to_nearest(
    coords_to_project: pd.DataFrame,
    coords_target: pd.DataFrame,
    lat_col: str = "latitude",
    lon_col: str = "longitude",
    return_distance: bool = True,
):
    """
    For each (lat, lon) in coords_to_project, find the nearest (lat, lon) in coords_target
    using great-circle (haversine) distance.

    Returns a DataFrame with:
      - lat, lon (from coords_to_project)
      - lat_nearest, lon_nearest (nearest coords from coords_target)
      - distance_m (optional; great-circle distance in meters)

    Notes:
      - Expects lat/lon in degrees.
      - Requires scikit-learn (BallTree). If not available, falls back to a slower NumPy version.
    """
    # Select and sanity-check inputs
    A = coords_to_project[[lat_col, lon_col]].to_numpy().astype(float)
    B = coords_target[[lat_col, lon_col]].to_numpy().astype(float)

    if A.size == 0 or B.size == 0:
        raise ValueError("Empty inputs: both coords_to_project and coords_target must have rows.")

    # Try fast haversine search via BallTree
    try:
        from sklearn.neighbors import BallTree

        # BallTree with haversine expects radians; distances are returned in radians
        A_rad = np.radians(A)
        B_rad = np.radians(B)
        tree = BallTree(B_rad, metric="haversine")
        dist_rad, ind = tree.query(A_rad, k=1)  # shape: (n, 1)

        # Convert outputs
        nearest = B[ind[:, 0]]  # back in degrees
        if return_distance:
            earth_radius_m = 6371000.0
            dist_m = (dist_rad[:, 0] * earth_radius_m)
        else:
            dist_m = None

    except Exception:
        # Fallback: vectorized NumPy (euclidean on degrees ≈ OK for tiny areas)
        # If your domain is larger, consider installing scikit-learn.
        A_exp = np.radians(A)[:, None, :]      # (n,1,2)
        B_exp = np.radians(B)[None, :, :]      # (1,m,2)

        # Haversine distance (vectorized)
        dlat = B_exp[..., 0] - A_exp[..., 0]
        dlon = B_exp[..., 1] - A_exp[..., 1]
        a = np.sin(dlat/2)**2 + np.cos(A_exp[..., 0]) * np.cos(B_exp[..., 0]) * np.sin(dlon/2)**2
        c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
        earth_radius_m = 6371000.0
        D = earth_radius_m * c  # (n, m)

        ind = np.argmin(D, axis=1)
        nearest = B[ind]
        dist_m = D[np.arange(D.shape[0]), ind] if return_distance else None

    # Build result table
    out = pd.DataFrame({
        lat_col: A[:, 0],
        lon_col: A[:, 1],
        "lat_nearest": nearest[:, 0],
        "lon_nearest": nearest[:, 1],
    })
    if return_distance:
        out["distance_m"] = dist_m

    return out


In [18]:
coords_prj_table = project_to_nearest(
    coords_to_project = sar_unq_coords,
    coords_target = df_unq_coords,
    lat_col="latitude",
    lon_col="longitude"
)

coords_prj_table = coords_prj_table.drop(columns = ['distance_m'])

In [19]:
print(len(sar_unq_coords))
print(len(df_unq_coords))
print(len(coords_prj_table))

4052
74
4052


In [20]:
gfw_sar = pd.merge(
    left = gfw_sar,
    right = coords_prj_table,
    how = 'left',
    on = spatial_keys
)

In [21]:
gfw_sar = gfw_sar.drop(columns=spatial_keys)
gfw_sar = gfw_sar.rename(columns={
    'lat_nearest': 'latitude',
    'lon_nearest': 'longitude'
})

In [22]:
print('sar len before merge:', sar_len_before)
print('sar len after merge:', len(gfw_sar))
gfw_sar.head()

sar len before merge: 4079
sar len after merge: 4079


,date,ssvid,speed_kn_inferred,heading_deg_inferred,length_m_inferred,presence_score,latitude,longitude
0,2024-05-27,218025010,3.03,309.488462,17.5,0.547,45.604168,13.625000
1,2024-05-09,218020360,0.63,194.323945,22.7,0.675,45.520832,13.541667
2,2024-05-09,None,1.44,357.043959,24.4,0.808,45.479168,13.458333
3,2024-05-09,None,3.98,354.179170,21.0,0.925,45.562500,13.625000
4,2024-05-09,203245488,0.24,76.203387,21.4,0.747,45.520832,13.541667


### Enrich df with sar data 

In [23]:
gfw_sar_grp = gfw_sar.groupby(keys).size().reset_index(name='sar_vessels_count')

In [24]:
gfw_sar_grp['date'] = gfw_sar_grp['date'].astype('datetime64[ns]') 

In [25]:
print(len(gfw_sar_grp))
print(len(df))
print('---')
df = pd.merge(
    left = df,
    right = gfw_sar_grp,
    how = 'left',
    on = keys
)
print(len(df))

1389
27084
---
27084


In [26]:
df['sar_vessels_count'] = df['sar_vessels_count'].fillna(0)

In [27]:
df.head()

,date,latitude,longitude,chl,thetao,sar_vessels_count
0,2024-01-01,45.479168,13.166667,0.432612,13.304394,0.0
1,2024-01-02,45.479168,13.166667,0.409951,13.248711,0.0
2,2024-01-03,45.479168,13.166667,0.387921,13.063446,0.0
3,2024-01-04,45.479168,13.166667,0.367288,12.772483,0.0
4,2024-01-05,45.479168,13.166667,0.347252,13.004619,0.0


### Enrich ais with target_lat, target_lon

In [28]:
ais_len_before = len(gfw_ais)

In [29]:
ais_unq_coords = gfw_ais.drop_duplicates(subset=spatial_keys)[spatial_keys].copy()

In [30]:
df_unq_coords = df.drop_duplicates(subset=spatial_keys)[spatial_keys].copy()

In [31]:
coords_prj_table = project_to_nearest(
    coords_to_project = ais_unq_coords,
    coords_target = df_unq_coords,
    lat_col="latitude",
    lon_col="longitude"
)

coords_prj_table = coords_prj_table.drop(columns = ['distance_m'])

In [32]:
print(len(ais_unq_coords))
print(len(df_unq_coords))
print(len(coords_prj_table))

687
74
687


In [33]:
gfw_ais = pd.merge(
    left = gfw_ais,
    right = coords_prj_table,
    how = 'left',
    on = spatial_keys
)

In [34]:
gfw_ais = gfw_ais.drop(columns=spatial_keys)
gfw_ais = gfw_ais.rename(columns={
    'lat_nearest': 'latitude',
    'lon_nearest': 'longitude'
})

In [35]:
print('ais len before merge:', ais_len_before)
print('ais len after merge:', len(gfw_ais))
gfw_ais.head()

ais len before merge: 8203
ais len after merge: 8203


,date,gear_type,mmsi,ship_name,hours,flag,latitude,longitude
0,2024-01-16,TRAWLERS,278527000,JEZ,0.607778,SVN,45.562500,13.625000
1,2024-06-17,TRAWLERS,278956000,141-KP LUSTURA,0.977778,SVN,45.604168,13.583333
2,2024-07-24,TRAWLERS,247050940,RINA S.,1.014167,ITA,45.645832,13.333333
3,2024-12-11,TRAWLERS,247030160,USODIMARE,0.325556,ITA,45.604168,13.416667
4,2024-07-07,TRAWLERS,247050940,RINA S.,0.961667,ITA,45.687500,13.250000


### Enrich df with ais data 

In [36]:
# total per (keys)
ais_totals = gfw_ais.groupby(keys).size().rename('ais_vessels_count')

# per-gear counts, including NaN as its own column
ais_gear_counts = (
    gfw_ais
    .groupby(keys + ['gear_type'], dropna=False)   # <-- keep NaNs
    .size()
    .unstack('gear_type', fill_value=0)
)

gfw_ais_grp = ais_totals.to_frame().join(ais_gear_counts).reset_index()


In [37]:
gfw_ais_grp.head(3)

,date,latitude,longitude,ais_vessels_count,DREDGE_FISHING,FISHING,OTHER_PURSE_SEINES,POTS_AND_TRAPS,SET_GILLNETS,TRAWLERS
0,2024-01-02,45.562500,13.666667,1,0,0,0,0,1,0
1,2024-01-02,45.604168,13.416667,1,0,0,0,0,0,1
2,2024-01-02,45.604168,13.500000,2,0,0,0,0,0,2


In [38]:
gfw_ais_grp['date'] = gfw_ais_grp['date'].astype('datetime64[ns]') 

In [39]:
print(len(gfw_ais_grp))
print(len(df))
print('---')
df = pd.merge(
    left = df,
    right = gfw_ais_grp,
    how = 'left',
    on = keys
)
print(len(df))

3176
27084
---
27084


In [40]:
df.head()

,date,latitude,longitude,chl,thetao,sar_vessels_count,ais_vessels_count,DREDGE_FISHING,FISHING,OTHER_PURSE_SEINES,POTS_AND_TRAPS,SET_GILLNETS,TRAWLERS
0,2024-01-01,45.479168,13.166667,0.432612,13.304394,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-01-02,45.479168,13.166667,0.409951,13.248711,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-01-03,45.479168,13.166667,0.387921,13.063446,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-01-04,45.479168,13.166667,0.367288,12.772483,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-01-05,45.479168,13.166667,0.347252,13.004619,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
for col in list(df.columns):
    df[col] = df[col].fillna(0)

In [42]:
df.head()

,date,latitude,longitude,chl,thetao,sar_vessels_count,ais_vessels_count,DREDGE_FISHING,FISHING,OTHER_PURSE_SEINES,POTS_AND_TRAPS,SET_GILLNETS,TRAWLERS
0,2024-01-01,45.479168,13.166667,0.432612,13.304394,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2024-01-02,45.479168,13.166667,0.409951,13.248711,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2024-01-03,45.479168,13.166667,0.387921,13.063446,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2024-01-04,45.479168,13.166667,0.367288,12.772483,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2024-01-05,45.479168,13.166667,0.347252,13.004619,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Export 

In [43]:
out_path = "../data/processed"
out_name = f"cpr_gfw_{year}.parquet"
out = os.path.join(out_path, out_name)

df.to_parquet(out, index=False)